In [ ]:
import os
from glob import glob
import json
import pickle
from datetime import datetime, timedelta

import avstack
import avapi

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

In [ ]:
def object_to_stone_soup_truth(obj, t_start):
    ts = t_start + timedelta(seconds=obj.t)
    xx, xy, xz = obj.position.x
    h, w, l = obj.box.size
    vx, vy, vz = obj.velocity.x
    er, ep, ey = obj.attitude.euler
    state = [xx, vx, xy, vy, xz, vz, h, w, l, er, ep, ey]
    metadata = {
        "object_type": obj.obj_type,
        "object_ID": obj.ID,
        "occlusion": obj.occlusion,
    }
    return GroundTruthState(state, timestamp=ts, metadata=metadata)


def update_truth_dictionary(truth, data_input, t_start):
    # get the set of visible objects
    objs_in_view = set()
    for agent in data_input["agent_data"]:
        for obj in data_input["agent_data"][agent]["objects"]["lidar"]:
            objs_in_view.add(obj.ID)

    # save all object states
    for obj in data_input["objects"]:
        if obj.ID not in truth["objects"]:
            truth["objects"][obj.ID] = GroundTruthPath()
        truth["objects"][obj.ID].append(object_to_stone_soup_truth(obj, t_start))
        # save visibility times
        if (obj.ID in objs_in_view) or (obj.ID > 10000):
            if obj.ID not in truth["visible_times"]["first"]:
                truth["visible_times"]["first"][obj.ID] = t_start + timedelta(
                    seconds=obj.t
                )
            truth["visible_times"]["last"][obj.ID] = t_start + timedelta(seconds=obj.t)

    return truth

### Process Truth Data

In [ ]:
from simulation.replayer import DatasetReplayer

log_dir = "last_run"

with open(os.path.join(log_dir, "metadata.json"), "r") as f:
    metadata = json.load(f)

In [ ]:
# Process truth data
t_start = datetime.fromtimestamp(metadata["t_start"])

truth = {
    "objects_visible": {},
    "visible_times": {"first": {}, "last": {}},
    "objects": {},
}

# load all truth objects
timestamps = []
replayer = DatasetReplayer(**metadata["replayer"])
for data_input in replayer(load_perception=False):
    truth = update_truth_dictionary(truth, data_input, t_start)
    timestamps.append(t_start + data_input["timestamp_dt"])

# massage set of visible objects
for obj_ID in truth["objects"]:
    if obj_ID in truth["visible_times"]["first"]:
        truth["objects_visible"][obj_ID] = GroundTruthPath()
        for state in truth["objects"][obj_ID]:
            if (
                truth["visible_times"]["first"][obj_ID]
                <= state.timestamp
                <= truth["visible_times"]["last"][obj_ID]
            ):
                truth["objects_visible"][obj_ID].append(state)

In [ ]:
from avstack.modules.perception.detections import DetectionContainerDecoder

# Process detection and track data
detections = {}
tracks = {}
for agent_subdir in [
    os.path.join(log_dir, "command-center"),
    *sorted(glob(os.path.join(log_dir, "agent-*"))),
]:
    agent_ID = int(agent_subdir[-1]) if "agent" in agent_subdir else "cc"
    detections[agent_ID] = []
    tracks[agent_ID] = []
    # detections
    for file in sorted(glob(os.path.join(agent_subdir, "detections", "*.txt"))):
        with open(file, "r") as f:
            data = json.load(f, cls=DetectionContainerDecoder)
        detections[agent_ID].append(data)

    # tracks
    for file in sorted(glob(os.path.join(agent_subdir, "tracks", "*.pickle"))):
        with open(file, "rb") as f:
            data = pickle.load(f)
        tracks[agent_ID].append(data)

In [ ]:
print(f"First replay timestamp: {timestamps[0]}\n")
print(f"First track timestamp: {list(tracks['cc'][-1].data)[0].states[0].timestamp}")

In [ ]:
from ordered_set import OrderedSet
from stonesoup.plotter import AnimatedPlotterly

plot_result = False

if plot_result:
    # plot in global
    truths_this = OrderedSet(truth["objects_visible"].values())
    plotter = AnimatedPlotterly(timestamps, tail_length=0.3, sim_duration=4)
    plotter.plot_ground_truths(truths_this, [0, 2])
    plotter.plot_tracks(
        tracks["cc"][-1].data, uncertainty=True, plot_history=True, mapping=[0, 2]
    )
    plotter.fig

In [ ]:
from ordered_set import OrderedSet
from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager
from stonesoup.measures import Euclidean
from stonesoup.metricgenerator.ospametric import OSPAMetric
from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric


tracking_filters = ["Baseline"]

ospa_generators = [
    OSPAMetric(
        c=40,
        p=1,
        generator_name=f"{tracking_filter} OSPA metrics",
        tracks_key=f"tracks_{tracking_filter}",
        truths_key="truths",
    )
    for tracking_filter in tracking_filters
]

siap_generators = [
    SIAPMetrics(
        position_measure=Euclidean((0, 2)),
        velocity_measure=Euclidean((1, 3)),
        generator_name=f"{tracking_filter} SIAP metrics",
        tracks_key=f"tracks_{tracking_filter}",
        truths_key="truths",
    )
    for tracking_filter in tracking_filters
]

uncertainty_generators = [
    SumofCovarianceNormsMetric(
        generator_name=f"{tracking_filter} OSPA metrics",
        tracks_key=f"tracks_{tracking_filter}",
    )
    for tracking_filter in tracking_filters
]
associator = TrackToTruth(association_threshold=30)

generators = ospa_generators + siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data(
    {
        "truths": OrderedSet(truth["objects_visible"].values()),
        "tracks_Baseline": tracks["cc"][-1].data,
    }
)
metrics = metric_manager.generate_metrics()

In [ ]:
from stonesoup.plotter import MetricPlotter


fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=["OSPA distances"])

In [ ]:
from stonesoup.metricgenerator.metrictables import SIAPTableGenerator


siap_metrics = metrics["Baseline SIAP metrics"]
siap_averages_baseline = {
    siap_metrics.get(metric)
    for metric in siap_metrics
    if metric.startswith("SIAP") and not metric.endswith(" at times")
}

_ = SIAPTableGenerator(siap_averages_baseline).compute_metric()
print("\n\nSIAP metrics for Baseline:")